In [ ]:
# OmniVoice against F5 on the same nine speakers, sentences, seed and reference
# clips - every configuration scored by one embedder and one recogniser here.
!pip -q install omnivoice speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

INPUT = Path("/kaggle/input")
print("mounted   :", sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else "nothing")

# Two runs died here, both on an assumed depth. The dataset does not mount at
# /kaggle/input/<slug> as version 1 assumed, nor one level down as version 2's
# glob assumed - it lands at /kaggle/input/datasets/<owner>/<slug>/. Searching
# for a file that has to be inside the repo makes the depth irrelevant.
# Version 2's failure listing was 40 corpus wavs and said nothing, so list
# directories instead.
found = sorted(INPUT.rglob("repo/scripts/benchmark.py"))
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo under /kaggle/input - see the directories above")
SOURCE = found[0].parent.parent
print("source    :", SOURCE)

REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(SOURCE, REPO)

# The worker runs out-of-process so it can hold its own pins on the laptop.
# Here one interpreter has everything, so point it at this one. The name is the
# flat one find_omnivoice_python() actually reads - version 3 set a nested
# VOICECLONE__ENGINES__OMNIVOICE__PYTHON, which nothing consumes, and every one
# of the 27 cells failed on an unavailable engine. It went unnoticed locally
# because the Mac has a prepared venv the lookup falls back to.
os.environ["VOICECLONE_OMNIVOICE_PYTHON"] = sys.executable

print("gpus      :", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("corpus    :", len(list((REPO / "eval/corpus").glob("*.wav"))), "clips")
for label in ("baseline", "finetuned", "finetuned-bestof4"):
    renders = list((REPO / "outputs/benchmark" / label).glob("*.wav"))
    print(f"renders   : {label:20s} {len(renders)}")
assert torch.cuda.device_count() >= 1, "no GPU - set the accelerator"


In [ ]:
# The control: score the F5 renders that were made on the laptop, with the
# embedder and recogniser running here. Nothing is re-rendered.
#
# It runs before OmniVoice so that a failing engine cannot cost us the control.
def run(*arguments):
    command = [sys.executable, "scripts/benchmark.py", *arguments]
    print(" ".join(command), flush=True)
    outcome = subprocess.run(command, cwd=REPO, text=True)
    print("exit code:", outcome.returncode, flush=True)
    return outcome.returncode

for label in ("baseline", "finetuned", "finetuned-bestof4"):
    assert run("--label", label + "-remote", "--score-only", "--replay-from", label,
               "--device", "cuda") == 0


In [ ]:
# Renders and scores OmniVoice. Same flags as the laptop run apart from the
# device, so the cells line up with every earlier run.
#
# This runs after the control on purpose. Version 3 failed here, and had the
# check below existed then it would have stopped the notebook before the
# control was written - which was the only part of that run worth keeping.
assert run("--label", "omnivoice", "--engine", "omnivoice", "--device", "cuda") == 0

# benchmark.py records failures in its results file, not in its exit code, so
# version 3 exited 0 with all 27 cells failed and the kernel reported COMPLETE.
# The file is the evidence; the exit code is not.
import json

summary = json.loads((REPO / "eval/results/omnivoice.json").read_text())["summary"]
print("omnivoice summary:", summary, flush=True)
failures = summary.get("failures") or 0
assert failures == 0, f"OmniVoice failed {failures} cells - see the traceback above"
assert (summary.get("renders") or 0) > 0, "OmniVoice produced no renders"


In [ ]:
outcome = subprocess.run(
    [sys.executable, "scripts/benchmark.py", "--compare",
     "baseline-remote", "finetuned-remote", "finetuned-bestof4-remote", "omnivoice"],
    cwd=REPO, text=True, capture_output=True,
)
print(outcome.stdout)
print(outcome.stderr[-2000:] if outcome.returncode else "", flush=True)


In [ ]:
# Everything worth bringing home: the results files and the renders.
import json

destination = Path("/kaggle/working/results")
destination.mkdir(exist_ok=True)
for path in (REPO / "eval/results").glob("*.json"):
    shutil.copy2(path, destination / path.name)

summary = {}
for name in ("omnivoice", "baseline-remote", "finetuned-remote", "finetuned-bestof4-remote"):
    path = destination / f"{name}.json"
    if path.is_file():
        data = json.loads(path.read_text())["summary"]
        summary[name] = {k: data.get(k) for k in ("held_out_mean", "cer_mean", "renders", "failures")}
print(json.dumps(summary, indent=2))

renders = Path("/kaggle/working/omnivoice_renders")
renders.mkdir(exist_ok=True)
for path in (REPO / "outputs/benchmark/omnivoice").glob("*.wav"):
    shutil.copy2(path, renders / path.name)
print("renders saved:", len(list(renders.glob("*.wav"))))
